# Accessing [Ensembl](https://www.ensembl.org/) with BioServices

Ensembl provides genome sequence and annotation for vertebrates and other
model organisms.  BioServices wraps its REST API so you can query genes,
transcripts, variants, and more from Python.

This notebook covers:

- [Introductory example](#introduction)
- [Archive](#archive)
- [Comparative genomics](#comparative)
- [Cross References](#reference)
- [Information](#information)
- [Lookup](#lookup)
- [Mapping](#mapping)
- [Ontology and Taxonomy](#ontology)
- [Overlap](#overlap)
- [Regulation](#regulation)
- [Sequences](#sequences)
- [Variation](#variation)

**References**: https://rest.ensembl.org/


In [1]:
from bioservices import Ensembl

e = Ensembl()


## <a name="introduction"></a> Introductory example

- Most of the methods take one or 2 compulsary arguments
- Some are just informative (the get_info family)
- An argument that is not part of the Ensembl API is **frmt**. It can be set to one of the Ensembl output format that is. The valid list of format
depends on the method. Those two are always available:
    - json
    - jsonp
- By default, output is in json format, which is transformed into a Python dictionary

In [2]:
res = e.get_archive('ENSG00000157764')
res

{'version': 15,
 'type': 'Gene',
 'assembly': 'GRCh38',
 'peptide': None,
 'possible_replacement': [],
 'latest': 'ENSG00000157764.15',
 'is_current': '1',
 'release': '115',
 'id': 'ENSG00000157764'}

In the following example, the format can be phyloxml

In [5]:
print(e.get_archive('ENSG00000157764'))

{'version': 15, 'is_current': '1', 'assembly': 'GRCh38', 'id': 'ENSG00000157764', 'release': '115', 'type': 'Gene', 'possible_replacement': [], 'latest': 'ENSG00000157764.15', 'peptide': None}


Here is another example where the requested frmt is json but there is a
parameter to specify the format (nh_format)

In [8]:
res = e.get_genetree_by_member_id('ENSG00000157764', 'human', frmt='json', 
                                  nh_format='phylip')
print(res[0:100])

<?xml version="1.0" encoding="UTF-8"?>

<phyloxml xsi:schemaLocation="http://www.phyloxml.org http:/


> Here, the input frmt (json) is changed since nh_format can be only in phyloxml format. When  a parameter specifies the format, it may overwrite the value of the argument **frmt** even if provided. 


In [9]:
# If your identifier is incorrect, you'll get a 500 error code 
# returned (most probably)
wrong = e.get_map_cds_to_region('ENST0000288602', '1..1000')
good = e.get_map_cds_to_region('ENST00000288602', '1..1000')
wrong, good['mappings'][0]


WARNING [bioservices.Ensembl:599]:  status is not ok with Internal Server Error


(500,
 {'coord_system': 'chromosome',
  'rank': 0,
  'seq_region_name': '7',
  'gap': 0,
  'start': 140924566,
  'strand': -1,
  'end': 140924703,
  'assembly_name': 'GRCh38'})

## <a name="archive"></a> Archive

In [10]:
# Get archived sequence given an identifer
archive = e.get_archive('ENSG00000157764')

In [11]:
identifiers = ["ENSG00000157764", "ENSG00000248378" ]
archives = e.post_archive(identifiers)
assert archive == archives[0]

## <a name="comparative"></a> Comparative genomics

### Gene tree by identifier

In [12]:
res = e.get_genetree_by_id('ENSGT00390000003602', nh_format='simple')
res['id'], res.keys()

('ENSGT00390000003602', dict_keys(['id', 'rooted', 'tree', 'type']))

In [13]:
res = e.get_genetree_by_id('ENSGT00390000003602', frmt='phyloxml')
print(res[0:200])

<?xml version="1.0" encoding="UTF-8"?>

<phyloxml xsi:schemaLocation="http://www.phyloxml.org http://www.phyloxml.org/1.10/phyloxml.xsd" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="ht


Retrieve genetree by member id and returns a phylip structure
This takes a few seconds and output xml is large`

In [15]:
# Here, the input frmt (json) is changed since nh_format can be 
# only in phyloxml format
res = e.get_genetree_by_member_id('ENSG00000157764', 'human', frmt='json', 
                                  nh_format='phylip')

In [16]:
len(res)

516777

In [17]:
print(res[0:500])

<?xml version="1.0" encoding="UTF-8"?>

<phyloxml xsi:schemaLocation="http://www.phyloxml.org http://www.phyloxml.org/1.10/phyloxml.xsd" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.phyloxml.org">
  <phylogeny type="gene tree" rooted="true">
    <clade branch_length="0">
      <taxonomy>
        <id>7742</id>
        <scientific_name>Vertebrata</scientific_name>
        <common_name>Vertebrates</common_name>
      </taxonomy>
      <clade branch_length="0">
        <co


In [18]:
res = e.get_genetree_by_member_symbol('human', 'BRCA2', 
                                      nh_format='simple')

In [19]:
print(res[0:200])

(((((((((((((((((((((ENSMOCP00000000295:0.069184,ENSPEMP00000001193:0.07127):0.006416,((ENSMAUP00000009408:0,ENSMAUP00000022423:0):0,ENSCGRP00015059417:0.022556):0.022641):0.016216,(((((ENSMSPP0001000


## <a name="reference"></a> Cross references

In [31]:
res = e.get_xrefs_by_id('ENST00000288602', 
                        all_levels=True)
res[0]

{'display_id': 'CCDS94218.1',
 'version': '1',
 'description': None,
 'dbname': 'CCDS',
 'db_display_name': 'CCDS',
 'info_text': '',
 'synonyms': [],
 'info_type': 'DIRECT',
 'primary_id': 'CCDS94218'}

In [32]:
res = e.get_xrefs_by_name('BRCA2', 'human')    
res[0]


{'description': 'BRCA2 DNA repair associated',
 'db_display_name': 'NCBI gene (formerly Entrezgene)',
 'primary_id': '675',
 'synonyms': [],
 'dbname': 'EntrezGene',
 'info_text': '',
 'version': '0',
 'info_type': 'DEPENDENT',
 'display_id': 'BRCA2'}

In [33]:
res = e.get_xrefs_by_symbol('BRCA2', 'homo_sapiens',
                            external_db='HGNC')    
res

[{'type': 'gene', 'id': 'ENSG00000139618'}, {'id': 'LRG_293', 'type': 'gene'}]

## <a name="information"></a> Information

In [34]:
len(e.get_info_analysis('human'))

249

In [35]:
e.get_info_assembly('human')['karyotype']

['1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 'X',
 'Y',
 'MT']

In [36]:
e.get_info_assembly_by_region('homo_sapiens', 'X')

{'assembly_exception_type': 'REF',
 'is_chromosome': 1,
 'is_circular': 0,
 'assembly_name': 'GRCh38',
 'length': 156040895,
 'coordinate_system': 'chromosome'}

In [37]:
len(e.get_info_biotypes('human'))

60

In [38]:
e.get_info_compara_methods()

{'SpeciesTree.species_tree_root': ['SPECIES_TREE'],
 'ProteinTree.protein_tree_node': ['PROTEIN_TREES'],
 'NCTree.nc_tree_node': ['NC_TREES'],
 'GenomicAlignTree.ancestral_alignment': ['EPO'],
 'ConstrainedElement.constrained_element': ['GERP_CONSTRAINED_ELEMENT'],
 'GenomicAlignTree.tree_alignment': ['EPO_EXTENDED'],
 'Homology.homology': ['ENSEMBL_PROJECTIONS',
  'ENSEMBL_ORTHOLOGUES',
  'ENSEMBL_PARALOGUES'],
 'ConservationScore.conservation_score': ['GERP_CONSERVATION_SCORE'],
 'GenomicAlignBlock.pairwise_alignment': ['LASTZ_NET', 'LASTZ_PATCH'],
 'SyntenyRegion.synteny': ['SYNTENY'],
 'GenomicAlignBlock.multiple_alignment': ['CACTUS_DB', 'PECAN', 'CACTUS_HAL']}

In [39]:
e.get_info_compara_by_method('EPO')[0]

{'method': 'EPO',
 'name': '10 primates EPO',
 'species_set': ['homo_sapiens',
  'chlorocebus_sabaeus',
  'nomascus_leucogenys',
  'microcebus_murinus',
  'gorilla_gorilla',
  'pan_paniscus',
  'pan_troglodytes',
  'macaca_mulatta',
  'macaca_fascicularis',
  'pongo_abelii'],
 'species_set_group': 'primates'}

In [40]:
e.get_info_comparas()

{'comparas': [{'release': 115, 'name': 'metazoa'},
  {'name': 'bacteria', 'release': 115},
  {'release': 115, 'name': 'pan_homology'},
  {'name': 'protists', 'release': 115},
  {'release': 115, 'name': 'vertebrates'},
  {'release': 115, 'name': 'plants'},
  {'name': 'fungi', 'release': 115}]}

In [41]:
e.get_info_data()

{'releases': [115]}

In [42]:
res = e.get_info_external_dbs('human')
[x['name'] for x in res if 'hgnc' in x['name'].lower()]

['HGNC', 'HGNC_trans_name']

In [43]:
e.get_info_ping()


1

In [44]:
e.get_info_rest()


{'release': '15.10'}

In [45]:
res = e.get_info_software()
res

{'release': 115}

In [46]:
res = e.get_info_species()
[x['name'] for x in res['species'] if 'ovis' in x['name']]


['ovis_aries',
 'ovis_aries_gca022416695v1',
 'ovis_aries_gca024222175v1',
 'ovis_aries_texel',
 'ovis_aries_gca022416775v1',
 'neovison_vison',
 'ovis_aries_gca022416915v1',
 'ovis_aries_gca011170295v1',
 'ovis_aries_gca022416685v1',
 'ovis_aries_gca022416755v1',
 'ovis_aries_gca022432825v1',
 'ovis_aries_gca018804185v1',
 'ovis_aries_gca022432835v1']

## <a name="lookup"></a> Lookup

In [47]:
res = e.get_lookup_by_id('ENSG00000157764', expand=True)
res.keys()

dict_keys(['id', 'logic_name', 'species', 'end', 'assembly_name', 'description', 'start', 'display_name', 'Transcript', 'object_type', 'canonical_transcript', 'version', 'biotype', 'source', 'seq_region_name', 'db_type', 'strand'])

In [48]:
res = e.post_lookup_by_id(["ENSG00000157764", "ENSG00000248378" ], 
                          expand=0)
res['ENSG00000157764']


{'version': 15,
 'description': 'B-Raf proto-oncogene, serine/threonine kinase [Source:HGNC Symbol;Acc:HGNC:1097]',
 'assembly_name': 'GRCh38',
 'source': 'ensembl_havana',
 'species': 'homo_sapiens',
 'biotype': 'protein_coding',
 'object_type': 'Gene',
 'start': 140719327,
 'strand': -1,
 'canonical_transcript': 'ENST00000646891.2',
 'end': 140924976,
 'db_type': 'core',
 'id': 'ENSG00000157764',
 'seq_region_name': '7',
 'logic_name': 'ensembl_havana_gene_homo_sapiens',
 'display_name': 'BRAF'}

In [49]:
res = e.get_lookup_by_symbol('homo_sapiens', 'BRCA2', expand=True)
len(res['Transcript'])

19

In [50]:
res = e.post_lookup_by_symbol('human', ["BRCA2", "BRAF" ], expand=True)
len(res['BRCA2']['Transcript'])

19

## <a name="mapping"></a> Mapping

	Description
- Convert from cDNA coordinates to genomic coordinates. Output reflects forward orientation coordinates as returned from the Ensembl API.
- GET map/cds/:id/:region 	Convert from CDS coordinates to genomic coordinates. Output reflects forward orientation coordinates as returned from the Ensembl API.
- GET map/:species/:asm_one/:region/:asm_two 	Convert the co-ordinates of one assembly to another
- GET map/translation/:id/:region 	Convert from protein (translation) coordinates to genomic coordinates. Output reflects forward orientation coordinates as returned from the Ensembl 

In [51]:
# the commented statement does not work
# res = e.get_map_assembly_one_to_two('GRCh37', 'NCBI36',
# region='X:10000000..1000100:1', species='human')
res = e.get_map_assembly_one_to_two('GRCh37', 'GRCh38', 
                                    region='X:1000000..1000100:1')
res

{'mappings': [{'original': {'start': 1000000,
    'coord_system': 'chromosome',
    'seq_region_name': 'X',
    'end': 1000100,
    'strand': 1,
    'assembly': 'GRCh37'},
   'mapped': {'coord_system': 'chromosome',
    'start': 1039265,
    'seq_region_name': 'X',
    'end': 1039365,
    'strand': 1,
    'assembly': 'GRCh38'}}]}

In [52]:
res = e.get_map_translation_to_region('ENSP00000288602', '100..300')
res['mappings'][0]  

{'seq_region_name': '7',
 'start': 140834609,
 'gap': 0,
 'rank': 0,
 'coord_system': 'chromosome',
 'end': 140834815,
 'strand': -1,
 'assembly_name': 'GRCh38'}

In [53]:
res = e.get_map_cds_to_region('ENST00000288602', '1..1000')
res['mappings'][0]

{'strand': -1,
 'assembly_name': 'GRCh38',
 'end': 140924703,
 'rank': 0,
 'start': 140924566,
 'gap': 0,
 'seq_region_name': '7',
 'coord_system': 'chromosome'}

In [54]:
res = e.get_map_cdna_to_region('ENST00000288602', '100..300')
res['mappings'][0]

{'seq_region_name': '7',
 'assembly_name': 'GRCh38',
 'strand': -1,
 'coord_system': 'chromosome',
 'gap': 0,
 'start': 140924566,
 'rank': 0,
 'end': 140924633}

## <a name="ontology"></a> Ontologies and Taxonomy

In [55]:
res = e.get_ontology_ancestors_by_id('GO:0005667')
res[0].keys()

dict_keys(['ontology', 'synonyms', 'name', 'namespace', 'subsets', 'definition', 'accession'])

In [56]:
res = e.get_ontology_ancestors_chart_by_id('GO:0005667')

In [57]:
res = e.get_ontology_descendants_by_id('GO:0005667')
res[0]['accession']

'GO:0032991'

In [58]:
res = e.get_ontology_by_id('GO:0005667')
res['accession']

'GO:0005667'

In [59]:
res = e.get_ontology_by_name('transcription factor complex')

In [60]:
res = e.get_taxonomy_classification_by_id(9606)
res[0]['children']

[{'leaf': 0,
  'id': '9606',
  'scientific_name': 'Homo sapiens',
  'name': 'Homo sapiens',
  'tags': {'genbank_common_name': ['human'],
   'name': ['Homo sapiens'],
   'authority': ['Homo sapiens Linnaeus, 1758'],
   'scientific_name': ['Homo sapiens']}}]

In [61]:
res = e.get_taxonomy_by_name('Homo')

In [62]:
e.get_taxonomy_by_id(9606)['scientific_name']

'Homo sapiens'

## <a name="overlap"></a> Overlap

In [63]:
res = e.get_overlap_by_id("ENSG00000157764", feature='gene')
len(res)

4

In [64]:
#ture=transcript;feature=cds;feature=exon
res = e.get_overlap_by_region('7:140424943-140624564', 
                              species='human', feature='gene')
len(res)

4

In [65]:
res = e.get_overlap_by_translation('ENSP00000288602', type='Superfamily')
len(res)

3

In [66]:
#feature=transcript_variation;contson;
res = e.get_overlap_by_translation('ENSP00000288602', 
                                   type='missense_variant',
                                   feature='transcript_variation')

len(res)

0

In [67]:
res = e.get_overlap_by_translation('ENSP00000288602', 
                                   type='missense_variant',
                                   feature='somatic_transcript_variation')

len(res)

0

## <a name="regulation"></a> Regulation

In [68]:
e.get_regulatory_by_id('ENSR00001348195', 'human')

WARNING [bioservices.Ensembl:599]:  status is not ok with Not Found


404

## <a name="sequences"></a> Sequences

In [69]:
sequence = e.get_sequence_by_id('ENSG00000157764', frmt='text')
print(sequence[0:120])

GTCTCCGCCCCCTTCCCCCGCTCCCCCCCGCACCCCCGCCTAGCGTCCTTCCCCCAATCCCCTCAGGCTCGGCTGCGCCCGGGGCCGCGGGCCGGTACCTGAGGTGGCCCAGGCGCCCTC


In [70]:
sequence = e.get_sequence_by_id('ENSG00000157764', frmt='fasta')
print(sequence[0:120])

>ENSG00000157764.15 chromosome:GRCh38:7:140719327:140924976:-1
GTCTCCGCCCCCTTCCCCCGCTCCCCCCCGCACCCCCGCCTAGCGTCCTTCCCCCAA


In [71]:
sequence = e.get_sequence_by_id('CCDS5863.1', frmt='fasta', 
                     object_type='transcript', db_type='otherfeatures',
                     type='cds', species='human')
print(sequence[0:120])

>CCDS5863.1.1
ATGGCGGCGCTGAGCGGTGGCGGTGGTGGCGGCGCGGAGCCGGGCCAGGCTCTGTTCAAC
GGGGACATGGAGCCCGAGGCCGGCGCCGGCGCCGGCGCCGCGGCC


In [72]:
sequence = e.get_sequence_by_id('ENSG00000157764', frmt='seqxml',
                                multiple_sequences=True,type='protein')
print(sequence[0:240])

<?xml version="1.0" encoding="UTF-8"?>

<seqXML xsi:noNamespaceSchemaLocation="http://www.seqxml.org/0.4/seqxml.xsd" seqXMLversion="0.4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <entry id="ENSP00000419060.2">
    <AAseq>MAAL


In [73]:
sequence = e.get_sequence_by_region('X:1000000..1000100:1', 'human')
sequence

{'seq': 'ctgtagaaacattagcctggctaacaaggtgaaaccccatctctactaacaatacaaaatattggttgggcgtggtggcgggtgcttgtaatcccagctac',
 'id': 'chromosome:GRCh38:X:1000000:1000100:1',
 'molecule': 'dna',
 'query': 'X:1000000..1000100:1'}

In [74]:
sequence = e.get_sequence_by_region('ABBA01004489.1:1..100', 'human',
                                    frmt='json', coord_system='seqlevel')
sequence

{'seq': 'ctgtactttccttgggatggagtagtttcgaaacacactttctgtagaatctgcaagtggatatttggacctgtctgaggaattcgttggaaacgggata',
 'query': 'ABBA01004489.1:1..100',
 'molecule': 'dna',
 'id': 'contig::ABBA01004489.1:1:100:1'}

## <a name=variation></a> Variation

In [75]:
e.get_variation_by_id('rs56116432', 'human')

{'var_class': 'SNP',
 'evidence': ['Frequency', '1000Genomes', 'ESP', 'ExAC', 'TOPMed', 'gnomAD'],
 'minor_allele': 'T',
 'synonyms': [],
 'most_severe_consequence': 'missense_variant',
 'mappings': [{'ancestral_allele': 'C',
   'end': 133256042,
   'coord_system': 'chromosome',
   'assembly_name': 'GRCh38',
   'seq_region_name': '9',
   'start': 133256042,
   'allele_string': 'C/A/T',
   'location': '9:133256042-133256042',
   'strand': 1},
  {'allele_string': 'C/A/T',
   'strand': 1,
   'location': 'HG2030_PATCH:82135-82135',
   'seq_region_name': 'HG2030_PATCH',
   'assembly_name': 'GRCh38',
   'start': 82135,
   'end': 82135,
   'coord_system': 'scaffold',
   'ancestral_allele': None}],
 'name': 'rs56116432',
 'MAF': 0.00274725,
 'ambiguity': 'H',
 'source': 'Variants (including SNPs and indels) imported from dbSNP'}

In [76]:
res = e.get_vep_by_id('COSM476', 'human')


In [77]:
res = e.get_vep_by_id('rs116035550', 'human')

In [78]:
res =e.get_vep_by_region('9:22125503-22125502:1', 'C', 'human')
res[0]['most_severe_consequence']

'intron_variant'